# DCASE 2026 Task 1 — Heterogeneous Audio Classification
## Baseline (HATR) — VSCode 실행 노트북 (Windows + RTX 3060)

**실행 전 체크리스트**
- VSCode에 **Jupyter 확장** (by Microsoft) 설치 확인
- CUDA 12.x 드라이버 설치 확인 (`nvidia-smi` 로 확인)
- 데이터셋 다운로드 필요: [BSD10k-v1.2](https://zenodo.org/records/17233904) / [BSD35k-CS](https://zenodo.org/records/19187099)

**디렉토리 구조**
```
DCASE_baseline/                        ← 최상위 루트 (PROJECT_DIR)
├── data/                              ← 데이터셋 폴더
│   ├── metadata/
│   │   ├── BSD10k_metadata.csv
│   │   ├── BSD35k-CS_metadata.csv
│   │   └── BST_description.csv
│   └── features/
│       ├── clap_audio_embeddings/
│       └── clap_text_embeddings/
└── dcase2026_task1_baseline/          ← git clone 위치 (코드 루트)
    ├── train_test.py
    ├── config.yaml
    └── ...
```

---

## 0. GPU 확인

In [2]:
import torch
print(torch.__version__)

2.5.1+cu121


In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  CUDA를 찾지 못했습니다. PyTorch CUDA 버전 설치를 확인하세요.')
    print('    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')

CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM: 8.6 GB


## 1. 패키지 설치

In [4]:
# 가상환경이 활성화된 상태에서 노트북을 실행하고 있는지 확인
import sys
print('Python 경로:', sys.executable)
# 출력에 .venv 경로가 포함되어 있어야 정상
# 예: C:\Users\yourname\DCASE_baseline\.venv\Scripts\python.exe

# 패키지 설치는 터미널에서 이미 완료했으므로 import 가능 여부만 확인
import torch, numpy, pandas, sklearn, yaml, matplotlib
print('모든 패키지 import 성공')

Python 경로: c:\Users\solok\Desktop\Dcase baseline\.venv\Scripts\python.exe
모든 패키지 import 성공


## 2. 경로 설정

**⚠️ `PROJECT_DIR`을 본인의 최상위 폴더 경로로 수정하세요.**

예시:
- `C:\Users\yourname\DCASE_baseline`
- `D:\projects\DCASE_baseline`

In [6]:
import os

# ↓↓↓ 여기만 본인 경로로 수정 ↓↓↓
PROJECT_DIR = r"C:\Users\solok\Desktop\Dcase baseline"  # 최상위 루트 (data/ 폴더가 있는 곳)
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑

CODE_DIR = os.path.join(PROJECT_DIR, 'dcase2026_task1_baseline')  # git clone 위치
DATA_ROOT = os.path.join(PROJECT_DIR, 'data')                     # 데이터셋 위치

# 폴더 자동 생성
os.makedirs(CODE_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print('최상위 루트  :', PROJECT_DIR)
print('코드 위치    :', CODE_DIR)
print('데이터 위치  :', DATA_ROOT)

최상위 루트  : C:\Users\solok\Desktop\Dcase baseline
코드 위치    : C:\Users\solok\Desktop\Dcase baseline\dcase2026_task1_baseline
데이터 위치  : C:\Users\solok\Desktop\Dcase baseline\data


## 3. 베이스라인 코드 세팅

> **방법 A**: GitHub에서 클론 (권장)  
> **방법 B**: 이미 clone 해둔 경우 스킵

In [7]:
# 방법 A: GitHub 클론 (처음 세팅 시)
# 이미 clone 되어 있으면 이 셀은 스킵해도 됩니다
os.chdir(PROJECT_DIR)
!git clone https://github.com/MTG/dcase2026_task1_baseline.git

os.chdir(CODE_DIR)
print('코드 위치:', os.getcwd())
!dir

코드 위치: C:\Users\solok\Desktop\Dcase baseline\dcase2026_task1_baseline


Cloning into 'dcase2026_task1_baseline'...


 C ����̺��� �������� �̸��� �����ϴ�.
 ���� �Ϸ� ��ȣ: 5C92-C3CC

 C:\Users\solok\Desktop\Dcase baseline\dcase2026_task1_baseline ���͸�

2026-04-14  ���� 06:16    <DIR>          .
2026-04-14  ���� 06:16    <DIR>          ..
2026-04-14  ���� 06:16                54 .gitignore
2026-04-14  ���� 06:16             4,529 build_dataset.py
2026-04-14  ���� 06:16             1,234 config.yaml
2026-04-14  ���� 06:16    <DIR>          data
2026-04-14  ���� 06:16             2,161 dataset_utils.py
2026-04-14  ���� 06:16            15,302 evaluate.py
2026-04-14  ���� 06:16               317 losses.py
2026-04-14  ���� 06:16               568 main.py
2026-04-14  ���� 06:16             7,691 models.py
2026-04-14  ���� 06:16             4,072 README.md
2026-04-14  ���� 06:16                82 requirements.txt
2026-04-14  ���� 06:16             2,139 summarize_results.py
2026-04-14  ���� 06:16            15,394 train_test.py
2026-04-14  ���� 06:16             3,260 utils.py
              13�� ����          

In [ ]:
# 방법 B: 이미 clone된 경우 — 작업 디렉토리만 이동
os.chdir(CODE_DIR)
print('작업 디렉토리:', os.getcwd())

## 4. config.yaml 경로 설정

`DATA_ROOT`는 위 셀에서 자동으로 설정되었습니다.  
`ACTIVE_DATASET`만 원하는 데이터셋으로 바꾸세요.

In [8]:
ACTIVE_DATASET = 'BSD10k-v1.2'  # 'BSD10k-v1.2' 또는 'BSD35k-CS'

# Windows 경로의 역슬래시를 슬래시로 변환 (yaml 파싱 안전)
data_root_str = DATA_ROOT.replace('\\', '/')

config_content = f"""# INPUT PATHS
active_dataset: {ACTIVE_DATASET}
datasets:
  BSD10k-v1.2:
    metadata_csv: "{data_root_str}/metadata/BSD10k_metadata.csv"
    audio_emb_folder: "{data_root_str}/features/clap_audio_embeddings"
    text_emb_folder: "{data_root_str}/features/clap_text_embeddings"
    class_names: "{data_root_str}/metadata/BST_description.csv"

  BSD35k-CS:
    metadata_csv: "{data_root_str}/metadata/BSD35k-CS_metadata.csv"
    audio_emb_folder: "{data_root_str}/features/clap_audio_embeddings"
    text_emb_folder: "{data_root_str}/features/clap_text_embeddings"
    class_names: "{data_root_str}/metadata/BST_description.csv"

# OUTPUT PATHS
output_path: "data"
processed_dataset_csv: "processed_dataset.csv"
class_dict_json: "class_dict.json"
top_class_dict_json: "top_class_dict.json"
top_class_subclass_dict_json: "top_class_subclass_dict.json"
"""

with open('config.yaml', 'w') as f:
    f.write(config_content)

print('config.yaml 저장 완료')
with open('config.yaml') as f:
    print(f.read())

config.yaml 저장 완료
# INPUT PATHS
active_dataset: BSD10k-v1.2
datasets:
  BSD10k-v1.2:
    metadata_csv: "C:/Users/solok/Desktop/Dcase baseline/data/metadata/BSD10k_metadata.csv"
    audio_emb_folder: "C:/Users/solok/Desktop/Dcase baseline/data/features/clap_audio_embeddings"
    text_emb_folder: "C:/Users/solok/Desktop/Dcase baseline/data/features/clap_text_embeddings"
    class_names: "C:/Users/solok/Desktop/Dcase baseline/data/metadata/BST_description.csv"

  BSD35k-CS:
    metadata_csv: "C:/Users/solok/Desktop/Dcase baseline/data/metadata/BSD35k-CS_metadata.csv"
    audio_emb_folder: "C:/Users/solok/Desktop/Dcase baseline/data/features/clap_audio_embeddings"
    text_emb_folder: "C:/Users/solok/Desktop/Dcase baseline/data/features/clap_text_embeddings"
    class_names: "C:/Users/solok/Desktop/Dcase baseline/data/metadata/BST_description.csv"

# OUTPUT PATHS
output_path: "data"
processed_dataset_csv: "processed_dataset.csv"
class_dict_json: "class_dict.json"
top_class_dict_json: "top_

## 5. 데이터셋 경로 확인

실제 파일이 있는지 확인합니다.

In [9]:
import os

checks = [
    os.path.join(DATA_ROOT, 'metadata', 'BSD10k_metadata.csv'),
    os.path.join(DATA_ROOT, 'features', 'clap_audio_embeddings'),
    os.path.join(DATA_ROOT, 'features', 'clap_text_embeddings'),
    os.path.join(DATA_ROOT, 'metadata', 'BST_description.csv'),
]

all_ok = True
for path in checks:
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ 없음!'
    print(f'{status}  {path}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n모든 경로 확인 완료. 다음 셀을 실행하세요.')
else:
    print('\n❌ 위 경로에 파일이 없습니다. 데이터셋을 올바른 위치에 배치했는지 확인하세요.')

❌ 없음!  C:\Users\solok\Desktop\Dcase baseline\data\metadata\BSD10k_metadata.csv
✅  C:\Users\solok\Desktop\Dcase baseline\data\features\clap_audio_embeddings
✅  C:\Users\solok\Desktop\Dcase baseline\data\features\clap_text_embeddings
❌ 없음!  C:\Users\solok\Desktop\Dcase baseline\data\metadata\BST_description.csv

❌ 위 경로에 파일이 없습니다. 데이터셋을 올바른 위치에 배치했는지 확인하세요.


In [ ]:
# metadata 폴더 파일 목록 확인
base = os.path.join(DATA_ROOT, 'metadata')
print('metadata 파일 목록:')
for f in os.listdir(base):
    print(f'  [{f}]')

## 6. train_test.py 버그 수정 패치

원본 코드에 `config.yaml`에 없는 키(`color_dict_path`)를 참조하는 버그가 있어 패치합니다.

In [ ]:
# train_test.py 패치
with open('train_test.py', 'r') as f:
    code = f.read()

code = code.replace(
    'color_dict_path = get_subconfig("color_dict_path")\n',
    '# color_dict_path removed (not in config)\n'
)
code = code.replace(
    'top_color_dict_path = get_subconfig("top_color_dict_path")\n',
    '# top_color_dict_path removed (not in config)\n'
)

with open('train_test.py', 'w') as f:
    f.write(code)

print('train_test.py 패치 완료')

# evaluate.py 패치
with open('evaluate.py', 'r') as f:
    code = f.read()

code = code.replace(
    'color_dict_path = get_subconfig("color_dict_path")\n',
    '# color_dict_path removed\n'
)
code = code.replace(
    'top_color_dict_path = get_subconfig("top_color_dict_path")\n',
    '# top_color_dict_path removed\n'
)

with open('evaluate.py', 'w') as f:
    f.write(code)

print('evaluate.py 패치 완료')

## 7. DataLoader num_workers 패치 (Windows 호환)

Windows에서 Jupyter 환경은 `if __name__ == '__main__'` 가드 없이 실행되므로  
`num_workers=0`으로 설정해야 멀티프로세싱 오류가 발생하지 않습니다.

In [ ]:
for fname in ['train_test.py', 'evaluate.py']:
    with open(fname, 'r') as f:
        code = f.read()
    # Windows Jupyter 환경: num_workers=0 이 안전
    code = code.replace('num_workers=4', 'num_workers=0')
    code = code.replace('num_workers=2', 'num_workers=0')
    with open(fname, 'w') as f:
        f.write(code)

print('num_workers 패치 완료 (→ 0, Windows 호환)')

## 8. 데이터셋 빌드

임베딩 파일 경로를 스캔하고 `processed_dataset.csv` 생성

In [ ]:
!python build_dataset.py

In [ ]:
# 빌드 결과 확인
import pandas as pd
df = pd.read_csv('data/processed_dataset.csv')
print(f'총 샘플 수: {len(df)}')
print(f'클래스 수: {df["class"].nunique()}')
print('\n클래스 분포:')
print(df['class'].value_counts().to_string())

## 9. 학습 실행

기본 설정: `both` (audio+text) + `audio` only 모드, 5-fold, 100 epoch  
RTX 3060 기준 약 **30~60분** 소요

In [ ]:
# 전체 파이프라인 실행 (학습 + 평가 + 요약)
!python train_test.py

## 10. 결과 요약

In [ ]:
!python summarize_results.py

with open('model_output/summary_metrics.txt') as f:
    print(f.read())

---
## 11. [선택] 빠른 단일 fold 테스트

전체 5-fold 실행 전에 1개 fold만 빠르게 돌려서 동작을 확인할 때 사용  
RTX 3060 기준 epoch=10 → 약 **2~5분**

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

# 현재 작업 경로 확인 (CODE_DIR 이어야 함)
print('현재 경로:', os.getcwd())

from utils import get_subconfig, set_seed, build_class_to_topclass_mapping
from models import BaseClassifier
from losses import CrossEntropyLoss
from dataset_utils import HATRDataset
from evaluate import evaluate_model
from train_test import train_model, init_weights

# ---- 설정 ----
FOLD_ID    = 0         # 테스트할 fold 번호
MODE       = 'both'    # 'both' | 'audio' | 'text'
NUM_EPOCHS = 10        # 빠른 테스트용 (실제는 100)
BATCH_SIZE = 64        # RTX 3060 12GB → 64 충분, OOM 시 32로 낮추기
HIDDEN     = 128
DROPOUT    = 0.1
K_FOLDS    = 5
# --------------

seed = set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

with open('data/class_dict.json') as f:
    class_dict = json.load(f)
with open('data/top_class_dict.json') as f:
    top_class_dict = json.load(f)

full_df = pd.read_csv('data/processed_dataset.csv')
labels  = full_df['class_idx'].tolist()
skf     = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)

for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    if fold != FOLD_ID:
        continue

    trainval_labels = [labels[i] for i in trainval_idx]
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx_rel, val_idx_rel = next(sss.split(np.zeros(len(trainval_labels)), trainval_labels))
    train_idx = [trainval_idx[i] for i in train_idx_rel]
    val_idx   = [trainval_idx[i] for i in val_idx_rel]

    train_df = full_df.iloc[train_idx].reset_index(drop=True)
    val_df   = full_df.iloc[val_idx].reset_index(drop=True)
    test_df  = full_df.iloc[test_idx].reset_index(drop=True)
    print(f'Fold {FOLD_ID}: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
    break

# Windows: num_workers=0 필수
train_loader = DataLoader(HATRDataset(train_df, aug=True,  mask_pct=0.7), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(HATRDataset(val_df,   aug=False),                batch_size=BATCH_SIZE, shuffle=False,                  num_workers=0, pin_memory=True)
test_loader  = DataLoader(HATRDataset(test_df,  aug=False),                batch_size=BATCH_SIZE, shuffle=False,                  num_workers=0, pin_memory=True)

emb_size_audio = 512 if MODE in ['audio', 'both'] else 0
emb_size_text  = 512 if MODE in ['text',  'both'] else 0

model = BaseClassifier(
    hidden_size=HIDDEN,
    num_classes=len(class_dict),
    emb_size_audio=emb_size_audio,
    emb_size_text=emb_size_text,
    dropout=DROPOUT,
    use_batch_norm=True,
    mode=MODE
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {total_params:,}')

init_weights(model)

out_dir = f'model_output/{MODE}/fold_{FOLD_ID}'
os.makedirs(out_dir, exist_ok=True)

best_acc, history, trained_model = train_model(
    model, train_loader, val_loader, device,
    num_epochs=NUM_EPOCHS,
    lr=0.001,
    classification_criterion=CrossEntropyLoss(),
    output_dir=out_dir,
    scheduler_type='step',
    patience=5,
    early_stopping_factor=3
)

print(f'\nBest val accuracy: {best_acc:.2f}%')

metrics = evaluate_model(
    BaseClassifier,
    os.path.join(out_dir, 'best_model.pth'),
    test_loader, device,
    class_to_topclass=build_class_to_topclass_mapping(class_dict, top_class_dict),
    output_dir=out_dir,
    fold_id=FOLD_ID,
    class_dict=class_dict
)

print('\n===== 최종 결과 =====')
for k, v in metrics.items():
    print(f'  {k}: {v:.2f}%')

## 12. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt
import json

with open(f'model_output/{MODE}/fold_0/history.json') as f:
    h = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(h.get('train_cls_loss', []), label='train CE loss', color='steelblue')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(h.get('val_accuracy', []), label='val accuracy', color='coral')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('%')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()
print('training_curve.png 저장 완료')

# Attention 가중치 변화 (mode='both'일 때)
if 'attention_audio' in h:
    fig2, ax2 = plt.subplots(figsize=(10, 3))
    audio_attn = [x if isinstance(x, float) else x[0] for x in h['attention_audio']]
    text_attn  = [x if isinstance(x, float) else x[0] for x in h['attention_text']]
    ax2.plot(audio_attn, label='audio weight α₁', color='steelblue')
    ax2.plot(text_attn,  label='text weight α₂',  color='coral')
    ax2.set_title('Attention Weights over Training')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(alpha=0.3)
    ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('attention_curve.png', dpi=150)
    plt.show()

## 13. 결과 로컬 백업

In [ ]:
import shutil

# model_output을 PROJECT_DIR 아래에 백업
backup_path = os.path.join(PROJECT_DIR, 'model_output_backup')
if os.path.exists(backup_path):
    shutil.rmtree(backup_path)
shutil.copytree('model_output', backup_path)
print(f'백업 완료: {backup_path}')

---
## 알려진 이슈 & 해결법

| 증상 | 원인 | 해결 |
|------|------|------|
| `KeyError: 'color_dict_path'` | config.yaml에 없는 키 참조 | 셀 6 패치 적용 |
| `RuntimeError: DataLoader worker` 오류 | Windows 멀티프로세싱 제한 | 셀 7 패치 (num_workers=0) |
| `CUDA out of memory` | batch_size 너무 큼 | `BATCH_SIZE=32` 로 줄이기 |
| `CUDA available: False` | PyTorch CPU 버전 설치됨 | `pip install torch ... --index-url https://download.pytorch.org/whl/cu121` |
| `Missing audio embedding` 많이 출력 | 임베딩 경로 불일치 | `DATA_ROOT` 경로 재확인, 셀 4 재실행 |
| `ModuleNotFoundError` | 작업 디렉토리가 CODE_DIR이 아님 | 셀 3-B 재실행 후 `os.chdir(CODE_DIR)` |

---
*DCASE 2026 Task 1 — Good luck!*